In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import joblib
from sklearn.metrics import precision_recall_curve, auc
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.optimizers import Adam, SGD
from keras.models import load_model
import xgboost as xgb
import lightgbm as lgb
import time

2025-08-20 11:19:46.860204: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-20 11:19:47.019823: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-08-20 11:19:47.119825: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-08-20 11:19:47.147426: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-08-20 11:19:47.267120: I tensorflow/core/platform/cpu_feature_guar

In [4]:
def prepare_data(data):
    X = data.drop(['sample', 'regulator', 'target', 'interaction', 'size'], axis=1).values
    y = data['interaction'].values
    X = np.asarray(X).astype(np.float32)
    y = np.asarray(y).astype(np.float32)
    return X, y

In [5]:
def load_size10():
    training_set = pd.read_csv(r'./caocao/training/size10.csv', index_col=0)
    valid_set = pd.read_csv(r'./caocao/validation/size10.csv', index_col=0)
    X_train, y_train = prepare_data(training_set)
    X_valid, y_valid = prepare_data(valid_set)
    del training_set
    del valid_set
    return X_train, y_train, X_valid, y_valid

In [6]:
def load_size50():
    '''
    Dataset contains all size50
    '''
    training_set_50_1 = pd.read_csv(r'./caocao/training/size50_10.csv', index_col=0)
    training_set_50_2 = pd.read_csv(r'./caocao/training/size50_20.csv', index_col=0)
 
    training_set_sum = pd.concat([training_set_50_1, training_set_50_2], ignore_index=True)

    del training_set_50_1
    del training_set_50_2

    training_set_sum = training_set_sum.dropna()
    X_train, y_train = prepare_data(training_set_sum)
    del training_set_sum
    valid_set = pd.read_csv(r'./caocao/validation/size50.csv', index_col=0)
    X_valid, y_valid = prepare_data(valid_set)
    del valid_set
    return X_train, y_train, X_valid, y_valid

In [7]:
def load_size70():
    '''
    Dataset contains all size 70
    '''
    
    training_set_70_1 = pd.read_csv(r'./caocao/training/size70_5.csv', index_col=0)
    training_set_70_2 = pd.read_csv(r'./caocao/training/size70_10.csv', index_col=0)
    training_set_70_3 = pd.read_csv(r'./caocao/training/size70_15.csv', index_col=0)
    training_set_70_4 = pd.read_csv(r'./caocao/training/size70_20.csv', index_col=0)

    
    training_set_sum = pd.concat([training_set_70_1, training_set_70_2,
                                 training_set_70_3, training_set_70_4], ignore_index=True)

    del training_set_70_1
    del training_set_70_2
    del training_set_70_3
    del training_set_70_4
    training_set_sum = training_set_sum.dropna()
    
    X_train, y_train = prepare_data(training_set_sum)
    del training_set_sum
    valid_set = pd.read_csv(r'./caocao/validation/size70.csv', index_col=0)
    X_valid, y_valid = prepare_data(valid_set)
    del valid_set
    return X_train, y_train, X_valid, y_valid

In [8]:
def load_size100():
    '''
    Dataset contains all size 100
    '''
    
    training_set_100_1 = pd.read_csv(r'./caocao/training/size100_5.csv', index_col=0)
    training_set_100_2 = pd.read_csv(r'./caocao/training/size100_10.csv', index_col=0)
    training_set_100_3 = pd.read_csv(r'./caocao/training/size100_15.csv', index_col=0)
    training_set_100_4 = pd.read_csv(r'./caocao/training/size100_20.csv', index_col=0)

    
    training_set_sum = pd.concat([training_set_100_1, training_set_100_2,
                                 training_set_100_3, training_set_100_4], ignore_index=True)

    del training_set_100_1
    del training_set_100_2
    del training_set_100_3
    del training_set_100_4
    training_set_sum = training_set_sum.dropna()
    
    X_train, y_train = prepare_data(training_set_sum)
    del training_set_sum
    valid_set = pd.read_csv(r'./caocao/validation/size100.csv', index_col=0)
    X_valid, y_valid = prepare_data(valid_set)
    del valid_set
    return X_train, y_train, X_valid, y_valid

In [9]:
def get_test_set(size):
    test_set_report = pd.read_csv(fr'./jump3_code/data for comparison/size{size}/size{size}.csv', index_col=0)
    X_test_report, y_test_report = prepare_data(test_set_report)
    del test_set_report
    return X_test_report, y_test_report

In [10]:
def get_report(model,report_filename, size, sample_number_max):
    edges = size*(size-1)
    start_predition_time = time.time()
    y_test_report_pred = model.predict(X_test)
    end_predition_time = time.time()
    with open(report_filename, 'w+') as f:
        for i in range(0, edges*sample_number_max*2, edges):
            precision1, recall1, _ = precision_recall_curve(y_test[i:i+edges], y_test_report_pred[i:i+edges])
            aupr1 = auc(recall1, precision1)
            f.write(f'{aupr1}\n')         
    return (end_predition_time-start_predition_time)*1000   

In [11]:
def get_report_xgb(model,report_filename, size, sample_number_max):
    edges = size*(size-1)
    dval = xgb.DMatrix(X_test, label=y_test)
    start_predition_time = time.time()
    y_test_report_pred = model.predict(dval)
    end_predition_time = time.time()
    with open(report_filename, 'w+') as f:
        for i in range(0, edges*sample_number_max*2, edges):
            precision1, recall1, _ = precision_recall_curve(y_test[i:i+edges], y_test_report_pred[i:i+edges])
            aupr1 = auc(recall1, precision1)
            f.write(f'{aupr1}\n')           
    return (end_predition_time-start_predition_time)*1000  

In [12]:
def get_report_lgb(model, report_filename, size, sample_number_max):
    edges = size*(size-1)
    start_predition_time = time.time()
    y_test_report_pred = model.predict(X_test, num_iteration=model.best_iteration)
    end_predition_time = time.time()
    with open(report_filename, 'w+') as f:
        for i in range(0, edges*sample_number_max*2, edges):
            precision1, recall1, _ = precision_recall_curve(y_test[i:i+edges], y_test_report_pred[i:i+edges])
            aupr1 = auc(recall1, precision1)
            f.write(f'{aupr1}\n')            
    return (end_predition_time-start_predition_time)*1000  

In [13]:
def run(time_path, size):
    if size==10:
        sample_number_max = 10
    else:
        sample_number_max = 4
    with open(time_path, 'w+') as f:        
        ##############################################################
        print(f'lr size {size} ...')
        lin_reg = LinearRegression()
        lr_start_training_time = time.time()
        lin_reg.fit(X_train, y_train)
        lr_end_training_time = time.time()
        f.write(f'{(lr_end_training_time - lr_start_training_time)*1000}\n')
        #joblib.dump(lin_reg, fr'./caocao/trained_model/size{model_size}_lr.pkl')
    
       # lr_prediction_time = get_report(lin_reg, f'./caocao/report_update/model{model_size}/linear_size10.txt', 10, 10)
       # lr_prediction_time = get_report(lin_reg, f'./caocao/report_update/model{model_size}/linear_size50.txt', 50, 4)
        lr_prediction_time = get_report(lin_reg, f'./caocao/report_single/linear_size{size}.txt', size, sample_number_max)
    
        f.write(f'{lr_prediction_time}\n')
        del lin_reg
        ##############################################################
        print(f'dt size {size}')
        dt_model = DecisionTreeClassifier(random_state=42)
        dt_start_training_time = time.time()
        dt_model.fit(X_train, y_train)
        dt_end_training_time = time.time()
        f.write(f'{(dt_end_training_time - dt_start_training_time)*1000}\n')
       # joblib.dump(dt_model, fr'./caocao/trained_model/size{model_size}_dt.pkl')
        #dt_prediction_time = get_report(dt_model, f'./caocao/report_update/model{model_size}/dt_size50.txt', 50, 4)
        #dt_prediction_time = get_report(dt_model, f'./caocao/report_update/model{model_size}/dt_size10.txt', 10, 10)
        dt_prediction_time = get_report(dt_model, f'./caocao/report_single/dt_size{size}.txt', size, sample_number_max)
        f.write(f'{dt_prediction_time}\n')
        del dt_model
        ##############################################################
        print(f'rf size {size}')
        rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
        rf_start_training_time = time.time()
        rf_model.fit(X_train, y_train)
        rf_end_training_time = time.time()
        f.write(f'{(rf_end_training_time - rf_start_training_time)*1000}\n')
        #joblib.dump(rf_model, fr'./caocao/trained_model/size{model_size}_rf.pkl')
        #rf_prediction_time = get_report(rf_model, f'./caocao/report_update/model{model_size}/rf_size50.txt', 50, 4)
        #rf_prediction_time = get_report(rf_model, f'./caocao/report_update/model{model_size}/rf_size10.txt', 10, 10)
        rf_prediction_time = get_report(rf_model, f'./caocao/report_single/rf_size{size}.txt', size, sample_number_max)
        f.write(f'{rf_prediction_time}\n')
        del rf_model
        ##############################################################
        print(f'knn size {size}')
        knn_model = KNeighborsClassifier(n_neighbors=10)
        # k=10 is the best
        # Train the model
        knn_start_training_time = time.time()
        knn_model.fit(X_train, y_train)
        knn_end_training_time = time.time()
        f.write(f'{(knn_end_training_time - knn_start_training_time)*1000}\n')
        #joblib.dump(knn_model, fr'./caocao/trained_model/size{model_size}_knn.pkl')
        #knn_prediction_time = get_report(knn_model, f'./caocao/report_update/model{model_size}/knn_size50.txt', 50, 4)
        #knn_prediction_time = get_report(knn_model, f'./caocao/report_update/model{model_size}/knn_size10.txt', 10, 10)
        knn_prediction_time = get_report(knn_model, f'./caocao/report_single/knn_size{size}.txt', size, sample_number_max)
        f.write(f'{knn_prediction_time}\n')
        del knn_model
        ##############################################################
        print(f'xgb size {size}')
        dtrain = xgb.DMatrix(X_train, label=y_train)
        dval = xgb.DMatrix(X_valid, label=y_valid)
        params = {
            'max_depth': 3,         # Maximum depth of a tree
            'eta': 0.1,             # Learning rate
            'objective': 'binary:logistic',  # Binary classification objective
            'eval_metric': 'logloss' # Evaluation metric
        }
        evallist = [(dtrain, 'train'), (dval, 'eval')]
        num_round = 100  # Number of boosting rounds
        xgb_start_training_time = time.time()
        xgb_model = xgb.train(params, dtrain, num_round, evals=evallist, early_stopping_rounds=10)
        xgb_end_training_time = time.time()
        f.write(f'{(xgb_end_training_time - xgb_start_training_time)*1000}\n')
        #xgb_prediction_time = xgb_model.save_model(fr'./caocao/trained_model/size{model_size}_xgb.pkl')
        #xgb_prediction_time = get_report_xgb(xgb_model, f'./caocao/report_update/model{model_size}/xgb_size10.txt', 10, 10)    
        #xgb_prediction_time = get_report_xgb(xgb_model, f'./caocao/report_update/model{model_size}/xgb_size50.txt', 50, 4)
        xgb_prediction_time = get_report_xgb(xgb_model, f'./caocao/report_single/xgb_size{size}.txt', size, sample_number_max)
        f.write(f'{xgb_prediction_time}\n')
        del xgb_model
        ##############################################################
        print(f'lgb size {size}')
        train_data = lgb.Dataset(X_train, label=y_train)
        val_data = lgb.Dataset(X_valid, label=y_valid, reference=train_data)
        params = {
            'boosting_type': 'gbdt',  # Gradient Boosting Decision Tree
            'objective': 'binary',    # Binary classification
            'metric': 'binary_logloss', # Metric to evaluate
            'num_leaves': 31,         # Maximum tree leaves for base learners
            'learning_rate': 0.05,    # Learning rate
            'feature_fraction': 0.9   # Fraction of features to be used for each tree
        }
        lgb_start_training_time = time.time()
        lgb_model = lgb.train(params, train_data, num_boost_round=100, valid_sets=[train_data, val_data])
        lgb_end_training_time = time.time()
        f.write(f'{(lgb_end_training_time - lgb_start_training_time)*1000}\n')
       # lgb_model.save_model(fr'./caocao/trained_model/size{model_size}_lgb.txt')
       # lgb_prediction_time = get_report_lgb(lgb_model, f'./caocao/report_update/model{model_size}/lgb_size10.txt', 10, 10)
        #lgb_prediction_time = get_report_lgb(lgb_model, f'./caocao/report_update/model{model_size}/lgb_size50.txt', 50, 4)
        lgb_prediction_time = get_report_lgb(lgb_model, f'./caocao/report_single/lgb_size{size}.txt', size, sample_number_max)
        f.write(f'{lgb_prediction_time}\n')
        del lgb_model
        ##############################################################
        print(f'nb size {size}')
        nb_classifier = GaussianNB()
        nb_start_training_time = time.time()
        nb_classifier.fit(X_train, y_train)
        nb_end_training_time = time.time()
        f.write(f'{(nb_end_training_time - nb_start_training_time)*1000}\n')
        #nb_prediction_time = get_report(nb_classifier, f'./caocao/report_update/model{model_size}/nb_size50.txt', 50, 4)
        #nb_prediction_time = get_report(nb_classifier, f'./caocao/report_update/model{model_size}/nb_size10.txt', 10, 10)
        nb_prediction_time = get_report(nb_classifier, f'./caocao/report_single/nb_size{size}.txt', size, sample_number_max)
        f.write(f'{nb_prediction_time}\n')
        del nb_classifier

In [ ]:
folder = 'report_single'
# X_train, y_train, X_valid, y_valid = load_size10()
# X_test, y_test = get_test_set(10)
# run(f'./caocao/{folder}/times_10.txt', 10)
# del X_train, y_train, X_valid, y_valid, X_test, y_test

# X_train, y_train, X_valid, y_valid = load_size50()
# X_test, y_test = get_test_set(50)
# run(f'./caocao/{folder}/times_50.txt', 50)
# del X_train, y_train, X_valid, y_valid, X_test, y_test

# X_train, y_train, X_valid, y_valid = load_size70()
# X_test, y_test = get_test_set(70)
# run(f'./caocao/{folder}/times_70.txt', 70)
# del X_train, y_train, X_valid, y_valid, X_test, y_test

X_train, y_train, X_valid, y_valid = load_size100()
X_test, y_test = get_test_set(100)
run(f'./caocao/{folder}/times_100.txt', 100)

lr size 100 ...
dt size 100
rf size 100
knn size 100
xgb size 100
[0]	train-logloss:0.17175	eval-logloss:0.17096
[1]	train-logloss:0.16317	eval-logloss:0.16229
[2]	train-logloss:0.15562	eval-logloss:0.15465
[3]	train-logloss:0.14899	eval-logloss:0.14796
[4]	train-logloss:0.14316	eval-logloss:0.14206
[5]	train-logloss:0.13803	eval-logloss:0.13688
[6]	train-logloss:0.13353	eval-logloss:0.13233
[7]	train-logloss:0.12959	eval-logloss:0.12835
[8]	train-logloss:0.12612	eval-logloss:0.12486
[9]	train-logloss:0.12309	eval-logloss:0.12180
[10]	train-logloss:0.12044	eval-logloss:0.11912
[11]	train-logloss:0.11811	eval-logloss:0.11676
[12]	train-logloss:0.11609	eval-logloss:0.11471
[13]	train-logloss:0.11432	eval-logloss:0.11293
[14]	train-logloss:0.11277	eval-logloss:0.11135
[15]	train-logloss:0.11143	eval-logloss:0.10998
[16]	train-logloss:0.11026	eval-logloss:0.10880
[17]	train-logloss:0.10924	eval-logloss:0.10778
[18]	train-logloss:0.10836	eval-logloss:0.10687
[19]	train-logloss:0.10759	eval-